In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import pickle
import sys
sys.path.append("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/")
from benchmarker import Benchmarker, read_sparse_h5, recompute_aggregate_scores

/mnt/datadisk/lizhongzhan/miniconda3/envs/benchmark_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/datadisk/lizhongzhan/miniconda3/envs/benchmark_env/lib/python3.10/site-packages/umap/__init__.py:9: ImportWarning: Tensorflow not installed; ParametricUMAP will be unavailable
  warn(


In [2]:
# data_dir = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/DATA/MISAR-seq/"
# # data_name = ["E11_s1", "E13_s1", "E15_s1", "E18_s1"]
# data_name = ["E15_s1", "E18_s1"]
# adatas_rna = []
# adatas_atac = []
# for name in data_name:
#     adata_rna = sc.read_h5ad(data_dir+f"/{name}/rna.h5ad")
#     adata_atac = sc.read_h5ad(data_dir+f"/{name}/atac.h5ad")
#     if name == "E11_s1":
#         prefix = "E11_0-S1"
#     else:
#         prefix = name.split("_")[0] + "_5-S1"
#     adata_rna.obs_names = [prefix+"#"+i for i in adata_rna.obs_names]
#     adata_atac.obs_names = [prefix+"#"+i for i in adata_atac.obs_names]
#     adata_rna.var_names_make_unique()
#     adata_atac.var_names_make_unique()
#     adata_rna.obs["sample"] = name
#     adata_atac.obs["sample"] = name
#     adatas_rna.append(adata_rna)
#     adatas_atac.append(adata_atac)

In [3]:
# annotation = pd.read_csv(data_dir+"/annotation.csv", index_col=0)
# annotation

In [4]:
# rna = sc.concat(adatas_rna)
# atac = sc.concat(adatas_atac)

In [5]:
# rna_annot = annotation.loc[rna.obs_names,]
# atac_annot = annotation.loc[atac.obs_names]

In [6]:
# (rna.obs_names == atac.obs_names).all()

In [7]:
# rna.obs["annotation"] = list(rna_annot["annotation"])
# atac.obs["annotation"] = list(rna_annot["annotation"])

In [8]:
# rna.write("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Multi_batch_embryo/rna.h5ad")
# atac.write("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Multi_batch_embryo/atac.h5ad")

In [9]:
import os
os.chdir("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Multi_batch_embryo/")

In [10]:
rna = sc.read_h5ad("rna.h5ad")
atac = sc.read_h5ad("atac.h5ad")

In [11]:
(rna.obs_names == atac.obs_names).all()

np.True_

In [12]:
atac.obs["sample"] = rna.obs["sample"].copy()

In [13]:
import anndata as ad
import h5py
import numpy as np
from scipy import sparse

def h5ad_to_h5(adata, output_file: str, batch_key: str):

    if adata.raw is not None and adata.raw.X is not None:
        X = adata.raw.X
        features = np.asarray(adata.raw.var_names, dtype=str)
    else:
        X = adata.X
        features = np.asarray(adata.var_names, dtype=str)

    barcodes = np.asarray(adata.obs_names, dtype=str)
    batches = np.asarray(adata.obs[batch_key], dtype=str)

    spatial = None
    if "spatial" in adata.obsm:
        spatial = np.asarray(adata.obsm["spatial"], dtype=np.float32)

    if sparse.issparse(X):
        X = X.tocsr()
    else:
        X = sparse.csr_matrix(X)

    if X.data.size == 0 or (np.all(X.data >= 0) and np.all(np.isclose(X.data, np.round(X.data)))):
        X.data = X.data.astype(np.int32, copy=False)
    else:
        X.data = X.data.astype(np.float32, copy=False)

    with h5py.File(output_file, "w") as f:
        g = f.create_group("matrix")

        g.create_dataset("data", data=X.data,
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("indices", data=X.indices.astype(np.int32, copy=False),
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("indptr", data=X.indptr.astype(np.int64, copy=False),
                         compression="gzip", compression_opts=4, shuffle=True)

        g.create_dataset("shape", data=np.asarray(X.shape, dtype=np.int64))

        g.create_dataset("barcodes", data=np.array(barcodes, dtype="S"))
        g.create_dataset("features", data=np.array(features, dtype="S"))
        g.create_dataset("batches", data=np.array(batches, dtype="S"))


        if spatial is not None:
            g.create_dataset(
                "spatial",
                data=spatial,
                compression="gzip",
                compression_opts=4,
                shuffle=True
            )

In [14]:
# h5ad_to_h5(rna, output_file="rna.h5", batch_key="sample")
# h5ad_to_h5(atac, output_file="atac.h5", batch_key="sample")

In [15]:
bm = Benchmarker(R_conda_env="Rbase")

Run evaluation methods

In [16]:
# data_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/datatset/MultiOmics/Multi_batch_embryo/"
# bm.run(methods=["MOFA2"],
#        RNA_file_path=data_folder+"rna.h5",
#        ATAC_file_path=data_folder+"/atac.h5",
#        n_cluster=15,
#        save_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Multi_batch_embryo/",
#        hvg_num=3000,
#        batch_key="batches"
#        )

In [17]:
# res = pd.read_csv("/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Multi_batch_embryo/multivi.csv", index_col=0)

In [18]:
# res.columns = ["UMAP1", "UMAP2", "cluster"]
# rna.obsm["X_umap"] = np.array(res[["UMAP1","UMAP2"]])
# rna.obs["cluster"] = [str(i) for i in list(res["cluster"])]

In [19]:
# sc.pl.umap(rna, color=["sample", "cluster"])
# rna1 = rna[rna.obs["sample"]=="E15_s1"]
# rna2 = rna[rna.obs["sample"]=="E18_s1"]
# sc.pl.spatial(rna1, spot_size=1, color=["cluster"])
# sc.pl.spatial(rna2, spot_size=1, color=["cluster"])

Plot

In [20]:
result_folder = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/result/multiOmicsPaired/Multi_batch_embryo/"
# methods = bm.multi_omics_methods.copy()
methods = ["scMDC", "Multigrate", "MultiVI", "MOFA2", "CellCharter", "PRESENT"]
res = bm.read_result(path=result_folder,
                     methods=methods,
                     reindex=False)

In [21]:
# metrics = bm.cal_metrics(adata=rna, batch_key="sample", label_key="annotation",
#                          res_dict=res, methods=methods, verbose=True, rep=1,
#                          min_max_scale=False,
#                          save=f"{result_folder}/metrics.pkl")

In [22]:
with open(f"{result_folder}/metrics.pkl", "rb") as f:
    metrics = pickle.load(f)
metric = metrics[0]

In [23]:
metric = recompute_aggregate_scores(metric)
metric["Total"][:-1] = metric["Batch correction"][:-1] * 0.4 + metric["Bio conservation"][:-1] * 0.6


In [24]:
figure_save_dir = "/mnt/datadisk/lizhongzhan/SpaMultiOmics/Benchmark/figures/Multi_omics_paired/Multi_batch_embryo"

In [25]:
bm.set_plot_params(params_dict={"figure.dpi": 300},
# font_file_path="/mnt/datadisk/lizhongzhan/SpaMultiOmics/Helvetica.ttf"
)

In [26]:
# bm.plot_heatmap(metric_df=metric, total_name="Total",
#                 save=f"{figure_save_dir}/summary_heatmap.pdf",
#                 # show_top=7,
#                 # show_bottom=0,
#                 # insert_marker_row = 8,
#                 )

In [27]:
from benchmarker import split_adata, transform_coord
import numpy as np
spatial = [i.obsm["spatial"] for i in split_adata(rna, batch_key="sample")]
spatial = transform_coord(spatial, vertical=False, axis="y", horizontal=False, angle=90)

In [28]:
spatial_methods = ["COSMOS", "CellCharter", "SpatialGlue", "PRESENT", "SMOPCA", "MISO"]
bg_dict = {i:"#D4B483" if i in spatial_methods else "#5873a4" for i in bm.all_methods }
bg_dict["RNA"] = "#5873a4"
bg_dict["ATAC"] = "#5873a4"
bg_dict["Protein"] = "#5873a4"
bg_dict["Batch"] = "#97a4af"
bg_dict["Cell type"] = "#97a4af"
bg_dict["E15"] = "#97a4af"
bg_dict["E18"] = "#97a4af"
bg_dict["Annotation"] = "#97a4af"


In [29]:
from benchmarker import get_scatter_cmap
palette = get_scatter_cmap([str(i) for i in list(range(15))])

In [30]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict=res["Cluster"],
#                 figsize=(12, 4.2),
#                 frameon=True,
#                 inner_gs_row=2,
#                 inner_gs_col=1,
#                 size=20,
#                 ncol=6,
#                 xlabel=["MultiVI", "PRESENT", "MOFA2",  "scMDC", "Multigrate", "CellCharter"],
#                 ylabel=None, #["E15", "E18"],
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 order=["MultiVI", "PRESENT", "MOFA2", "scMDC", "Multigrate", "CellCharter"],
#                 outer_row_hspace=0,
#                 outer_col_wspace=0.02,
#                 inner_common_camp=True,
#                 palette=palette,
#                 # inner_col_wspace = -0.12,
#                 ylabel_pad = 0.02,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.013, save_dpi=600,
#                 save=f"{figure_save_dir}/spatial_methods.pdf",
#                 rasterized=True,
#                 )

In [31]:
res["Batch"] = {}
for m in methods:
    res["Batch"][m] = np.array(rna.obs["sample"]).astype(str)

In [32]:
# bm.plot_umap(embed_dict=res["UMAP"],
#              batch_dict=res["Batch"],
#              annot_list=list(rna.obs["annotation"]),
#              figsize=(12, 4.2),
#              frameon=True,
#              inner_gs_row=2,
#              inner_gs_col=1,
#              size=10,
#              ncol=6,
#              xlabel=["MultiVI", "PRESENT", "MOFA2",  "scMDC", "Multigrate", "CellCharter"],
#              only_show_top=False,
#              ylabel=["Batch", "Cell type"],
#              only_show_left=True,
#              background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#              order=["MultiVI", "PRESENT", "MOFA2",  "scMDC", "Multigrate", "CellCharter"],
#              axis_width=1.2,
#              axis_color="lightgrey",
#              outer_col_wspace=0.05,
#              save_dpi=600,
#              ylabel_pad=0.0168,
#              xlabel_pad=0.013,
#              outer_row_hspace=0.22,
#              merge=True,
#              merge_margin_size=0.4,
#              palettes=[None, palette_annot],
#              save=f"{figure_save_dir}/umap_methods.pdf"
# )

In [41]:
# bm.plot_spatial(spatial=spatial,
#                 label_dict={"annot": np.array(rna.obs["annotation"]).reshape(-1,1)},
#                 figsize=(1.97, 4.2),
#                 frameon=True,
#                 inner_gs_row=2, inner_gs_col=1,
#                 size=20,
#                 ncol=1,
#                 xlabel=["Annotation"],
#                 ylabel=["E15", "E18"],
#                 only_show_left=True,
#                 axis_width = 1.2,
#                 axis_color="lightgrey",
#                 outer_row_hspace=0.15,
#                 outer_col_wspace=0.1,
#                 background_color = lambda x: bg_dict[x] if x in bg_dict else None,
#                 xlabel_pad=0.013,
#                 ylabel_pad=0.0165,
#                 save_dpi=600,
#                 palette=palette,
#                 # save=f"{figure_save_dir}/spatial_annot.pdf"
#                 )

In [34]:
palette_annot = {}
for i in range(15):
    palette_annot[sorted(set(rna.obs["annotation"]))[i]] = get_scatter_cmap([str(i) for i in list(range(15))])[str(i)]

In [42]:
# bm.plot_legend(palette_annot, marker="o", ncol=1,
# # save=f"{figure_save_dir}/annot_legend.pdf",
# )